# 09 附錄：舊版程式碼

**用途**：舊版樹狀圖寫法與 pivot 匯出，僅供對照。

| 項目 | 說明 |
|---|---|
| 輸入 | data/raw CSV |
| 輸出 | pivot_table.csv（存於 notebooks/_scratch/） |
| 來源 | MVA 期末報告分析.ipynb（Colab 原檔，191 cells） 之 cell 184–190（Update 05/29 最終版） |

> 歷史程式碼，變數可能依賴舊版流程，不保證可獨立執行。


In [ ]:
# 共用設定：載入資料並提供 df / pivot_table_age_std / pivot_table_cleaned / scaled_data / taiwan_map / prop 等變數
# （取代原 Colab 的 Drive 掛載、字型下載與 GDAL 設定；詳見 _legacy_setup.py）
import sys; sys.path.insert(0, '..')   # _legacy_setup.py 位於 notebooks/
from _legacy_setup import *

In [ ]:
# 前置狀態（原 Colab 於 06 產生）
scaled_clustering_data = scaled_data

## 附錄

* Some old version code

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram

# 1. 準備必要數據
county_names = pivot_table_cleaned.index.tolist()

# 2. 計算兩種方法在多維空間中的 Linkage 矩陣
# (Ward 法與 Average 法會依據不同的距離演算法建立階層樹)
Z_ward = linkage(scaled_clustering_data, method='ward')
Z_average = linkage(scaled_clustering_data, method='average')

# 確保全域中文字型已設定
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# ==========================================================
# 📊 圖片一：Ward's Method 樹狀圖
# ==========================================================
plt.figure(figsize=(14, 6))

# ✨ 核心技巧：動態計算 k=5 的顏色切分門檻
# Z 矩陣的第三欄 [-4] 代表倒數第 4 次融合的高度，低於此高度剛好會切出 5 個群體
threshold_ward = Z_ward[-4, 2]

dendrogram(
    Z_ward,
    labels=county_names,
    color_threshold=threshold_ward,  # 自動讓 5 個群體染上不同顏色
    leaf_rotation=45,                # 縣市名稱旋轉 45 度避免重疊
    leaf_font_size=12
)

# 畫出視覺輔助切線
plt.axhline(y=threshold_ward, color='#e06666', linestyle='--', linewidth=1.5,
            label=f'k=5 Cutoff Line (Height: {threshold_ward:.2f})')

plt.title("Hierarchical Clustering Dendrogram (Ward's Method, k=5)", fontsize=16, pad=15)
plt.xlabel('Counties / Cities', fontsize=12)
plt.ylabel('Cluster Distance (Inertia)', fontsize=12)
plt.grid(axis='y', linestyle=':', alpha=0.5)
plt.legend(loc='upper right')
plt.tight_layout()

# 💾 儲存第一張圖
plt.savefig('dendrogram_ward_k5.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ==========================================================
# 📊 圖片二：Average Linkage Method 樹狀圖
# ==========================================================
plt.figure(figsize=(14, 6))

# 同理，計算 Average 法在 k=5 時的切分高度
threshold_avg = Z_average[-4, 2]

dendrogram(
    Z_average,
    labels=county_names,
    color_threshold=threshold_avg,   # 自動讓 5 個群體染上不同顏色
    leaf_rotation=45,
    leaf_font_size=12
)

# 畫出視覺輔助切線
plt.axhline(y=threshold_avg, color='#e06666', linestyle='--', linewidth=1.5,
            label=f'k=5 Cutoff Line (Height: {threshold_avg:.2f})')

plt.title("Hierarchical Clustering Dendrogram (Average Method, k=5)", fontsize=16, pad=15)
plt.xlabel('Counties / Cities', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.grid(axis='y', linestyle=':', alpha=0.5)
plt.legend(loc='upper right')
plt.tight_layout()

# 💾 儲存第二張圖
plt.savefig('dendrogram_average_k5.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv(str(CSV_PATH))
print(f"CSV file loaded successfully")
display(df.head())

In [ ]:
df_filtered = df[(df['性別'] == '全') &(df['縣市別'] != '全國') & (df['癌症別'] == '肺、支氣管及氣管')]
display(df_filtered)

In [ ]:
pivot_table_age_std = df_filtered.pivot_table(
    index='縣市別',
    columns='癌症診斷年',
    values='年齡標準化發生率  WHO 2000世界標準人口 (每10萬人口)',
    aggfunc='mean' # Assuming you want the mean if there are multiple entries
)
display(pivot_table_age_std)

In [ ]:
pivot_table_age_std.to_csv('pivot_table.csv', na_rep="NA", encoding='utf-8-sig')